# Exploratory Analysis & Insight Generation

**Workforce Intelligence and Organizational Performance Analysis**  
Roblox Africa Operations (Ghana Hub)

Stage 7 &nbsp;·&nbsp; Working step, not a submitted deliverable &nbsp;·&nbsp; 4 August 2026

---

### What this notebook does

Answers the five secondary objectives from the capstone brief, one section each, using the
database built in Stage 6. Every section follows the same shape:

1. The SQL query
2. A chart
3. One plain-English sentence that **compares something to something** — not just a number on
   its own

### Why this notebook exists

It is not itself a deliverable. It is what produces deliverable 3, the *insight driven analysis
report*. The findings written here get copied into that report in Stage 9, with the charts
alongside them.

### Source

All queries run against `roblox_workforce.db`, built and verified in Stage 6. `master_employee`
is the view joining the five person-level tables. `employee_performance` and
`department_performance` are queried separately — aggregated first, joined second — exactly as
the Stage 6 warnings require.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DB = "../05_Database/roblox_workforce.db"
con = sqlite3.connect(DB)

plt.rcParams['figure.dpi'] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

findings = []   # every plain-English sentence gets collected here, printed at the end

def finding(text):
    findings.append(text)
    print("FINDING:", text)

print("connected:", DB)

---
## Objective 1 &nbsp;—&nbsp; Workforce composition

> Brief: *“Analyze workforce composition by department, role, and gender.”*

Four breakdowns: department, position, gender, employment status. Age band is added as a fifth
because the brief's own “Expected insights” section separately lists age distribution.

In [ ]:
by_dept = pd.read_sql("""
    SELECT d.department_name, COUNT(*) AS headcount
    FROM employee e JOIN department d ON e.department_code = d.department_code
    GROUP BY d.department_name ORDER BY headcount DESC
""", con)

by_dept["share"] = (by_dept["headcount"] / by_dept["headcount"].sum() * 100).round(1)
by_dept

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(by_dept["department_name"], by_dept["headcount"], color="#4361EE")
ax.invert_yaxis()
ax.set_xlabel("Employees"); ax.set_title("Headcount by department")
plt.tight_layout(); plt.show()

In [ ]:
top = by_dept.iloc[0]; bottom = by_dept.iloc[-1]
finding(f"{top.department_name} is the largest department at {top.headcount:,} people "
        f"({top.share}% of headcount), only {top.headcount - bottom.headcount} more than "
        f"{bottom.department_name}, the smallest at {bottom.headcount:,} — headcount is spread "
        f"almost evenly across all 8 departments.")

In [ ]:
by_position = pd.read_sql("""
    SELECT position, COUNT(*) AS headcount FROM employee
    GROUP BY position ORDER BY headcount DESC
""", con)
by_position["share"] = (by_position["headcount"] / by_position["headcount"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(by_position["position"], by_position["headcount"], color="#7B2CBF")
ax.set_ylabel("Employees"); ax.set_title("Headcount by position")
plt.tight_layout(); plt.show()

top = by_position.iloc[0]
finding(f"{top.position} is the single largest role at {top.share}% of the workforce "
        f"({top.headcount:,} people) — more than any other position, including Manager.")

In [ ]:
by_gender = pd.read_sql("SELECT gender, COUNT(*) AS n FROM employee GROUP BY gender", con)
by_gender["share"] = (by_gender["n"] / by_gender["n"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(4,4))
ax.pie(by_gender["n"], labels=by_gender["gender"], autopct="%1.0f%%",
       colors=["#4361EE","#EF476F"])
ax.set_title("Gender split")
plt.tight_layout(); plt.show()

m, f = by_gender.set_index("gender").loc[["Male","Female"], "share"]
finding(f"The workforce is {m}% male and {f}% female — a gap of {round(m-f,1)} points, "
        f"fairly close to balanced but not even.")

In [ ]:
by_status = pd.read_sql("SELECT employee_status, COUNT(*) AS n FROM employee GROUP BY employee_status", con)
by_status["share"] = (by_status["n"] / by_status["n"].sum() * 100).round(1)
by_status

In [ ]:
active = by_status.set_index("employee_status").loc["Active"]
not_active = by_status[by_status.employee_status != "Active"]["n"].sum()
finding(f"{active.share}% of employees are Active. The remaining {round(100-active.share,1)}% "
        f"({not_active:,} people) are Inactive or On Leave — worth knowing before headcount is "
        f"used as a proxy for working capacity.")

In [ ]:
age_bands = pd.read_sql("""
    SELECT
        CASE WHEN age < 25 THEN 'Under 25'
             WHEN age < 35 THEN '25-34'
             WHEN age < 45 THEN '35-44'
             WHEN age < 55 THEN '45-54'
             ELSE '55 and over' END AS age_band,
        COUNT(*) AS n
    FROM employee GROUP BY age_band
""", con)
order = ["Under 25","25-34","35-44","45-54","55 and over"]
age_bands = age_bands.set_index("age_band").loc[order].reset_index()
age_bands["share"] = (age_bands["n"] / age_bands["n"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(age_bands["age_band"], age_bands["n"], color="#06A77D")
ax.set_ylabel("Employees"); ax.set_title("Headcount by age band")
plt.tight_layout(); plt.show()

smallest = age_bands.loc[age_bands["n"].idxmin()]
finding(f"{smallest.age_band} is the smallest age band at only {smallest.share}% of the "
        f"workforce — the other four bands are all above {age_bands[age_bands.age_band!=smallest.age_band].share.min()}%.")

---
## Objective 2 &nbsp;—&nbsp; Education versus job placement

> Brief: *“Evaluate education level and field of study against job placement.”*

**An assumption is required here, and it is stated rather than hidden.** The data does not say
which field of study ‘belongs’ to which department — that judgement has to be supplied. The
mapping below is the most defensible reading of the 8 fields against the 8 departments, but a
different, equally reasonable mapping would shift the mismatch numbers. Treat this section as a
first estimate, and confirm the mapping logic with the client before it goes in the final report.

In [ ]:
# The assumption, stated explicitly:
# each field of study is treated as a 'match' only for the department(s) it most obviously feeds.
# Every other pairing counts as a mismatch. This is a judgement call, not a fact in the data.
FIELD_MATCHES_DEPARTMENT = {
    "Computer Science":               ["Engineering Department", "Data and Analytics Department"],
    "Information Technology":         ["Engineering Department", "Data and Analytics Department"],
    "Information Technology & Law":   ["Engineering Department", "Data and Analytics Department"],
    "Physics":                        ["Engineering Department", "Data and Analytics Department"],
    "Data Analytics":                 ["Data and Analytics Department"],
    "Economics":                      ["Finance Department"],
    "Business Administration":        ["Product Management Department", "Operations Department",
                                        "Human Resources Department", "Marketing Department"],
    "Engineering":                    ["Engineering Department"],
}

placement = pd.read_sql("""
    SELECT ed.field_of_study, d.department_name, COUNT(*) AS n
    FROM employee e
    JOIN education ed ON e.employee_id = ed.employee_id
    JOIN department d ON e.department_code = d.department_code
    GROUP BY ed.field_of_study, d.department_name
""", con)

placement["matched"] = placement.apply(
    lambda r: r.department_name in FIELD_MATCHES_DEPARTMENT.get(r.field_of_study, []), axis=1)

pivot = placement.pivot_table(index="field_of_study", columns="department_name",
                               values="n", fill_value=0, aggfunc="sum")
pivot

In [ ]:
total = placement["n"].sum()
matched = placement[placement.matched]["n"].sum()
mismatched = total - matched

fig, ax = plt.subplots(figsize=(4,4))
ax.pie([matched, mismatched], labels=["Matched field", "Mismatched field"],
       autopct="%1.0f%%", colors=["#06A77D","#EF476F"])
ax.set_title("Education field vs. actual department")
plt.tight_layout(); plt.show()

finding(f"Under the mapping used here, {round(mismatched/total*100,1)}% of employees "
        f"({mismatched:,} of {total:,}) work in a department that does not obviously match "
        f"their field of study. This is sensitive to the mapping assumption above and should "
        f"be confirmed with the client before being treated as a firm number.")

In [ ]:
by_field = placement.groupby("field_of_study").apply(
    lambda g: pd.Series({"total": g["n"].sum(), "matched": g[g.matched]["n"].sum()})
).reset_index()
by_field["match_rate"] = (by_field["matched"] / by_field["total"] * 100).round(1)
by_field = by_field.sort_values("match_rate")

worst = by_field.iloc[0]
finding(f"{worst.field_of_study} graduates have the lowest placement match at "
        f"{worst.match_rate}%, against a workforce mismatch rate of "
        f"{round(mismatched/total*100,1)}% overall.")

---
## Objective 3 &nbsp;—&nbsp; Compensation and cost

> Brief: *“Assess compensation patterns and cost distribution.”*

**Every figure below is USD, per the client's confirmation — and every figure is per period as
supplied**, because the client has not confirmed whether Basic Salary and Allowances are monthly
or annual (open question O1). Nothing here is annualised or multiplied.

In [ ]:
comp_dept = pd.read_sql("""
    SELECT d.department_name,
           COUNT(*) AS headcount,
           SUM(f.basic_salary + f.allowances) AS total_compensation,
           ROUND(AVG(f.basic_salary + f.allowances), 2) AS avg_compensation
    FROM employee e
    JOIN department d ON e.department_code = d.department_code
    JOIN finance f ON e.employee_id = f.employee_id
    GROUP BY d.department_name ORDER BY total_compensation DESC
""", con)
comp_dept["cost_share"] = (comp_dept["total_compensation"] / comp_dept["total_compensation"].sum() * 100).round(1)
comp_dept["headcount_share"] = (comp_dept["headcount"] / comp_dept["headcount"].sum() * 100).round(1)
comp_dept

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(comp_dept["department_name"], comp_dept["total_compensation"], color="#E0A200")
ax.invert_yaxis()
ax.set_xlabel("Total compensation, USD per period as supplied")
ax.set_title("Total compensation by department")
plt.tight_layout(); plt.show()

# the actual gap between a department's share of headcount and its share of cost
comp_dept["share_gap"] = (comp_dept["cost_share"] - comp_dept["headcount_share"]).round(1)
gap = comp_dept.loc[comp_dept["share_gap"].abs().idxmax()]
if abs(gap.share_gap) < 0.2:
    finding(f"Compensation share tracks headcount share almost exactly across all 8 "
            f"departments — the largest gap is only {abs(gap.share_gap)} points "
            f"({gap.department_name}), so no department is paid disproportionately to its size.")
else:
    direction = "more" if gap.share_gap > 0 else "less"
    finding(f"{gap.department_name} holds {gap.headcount_share}% of headcount but "
            f"{gap.cost_share}% of total compensation — {abs(gap.share_gap)} points {direction} "
            f"cost share than headcount share, the largest gap of any department.")

In [ ]:
comp_pos = pd.read_sql("""
    SELECT e.position,
           ROUND(AVG(f.basic_salary + f.allowances), 2) AS avg_compensation
    FROM employee e JOIN finance f ON e.employee_id = f.employee_id
    GROUP BY e.position ORDER BY avg_compensation DESC
""", con)

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(comp_pos["position"], comp_pos["avg_compensation"], color="#4361EE")
ax.set_ylabel("Average compensation, USD per period as supplied")
ax.set_title("Average compensation by position")
plt.tight_layout(); plt.show()

hi, lo = comp_pos.iloc[0], comp_pos.iloc[-1]
pct_diff = round((hi.avg_compensation / lo.avg_compensation - 1) * 100, 1)
if abs(pct_diff) < 2:
    finding(f"Average compensation is almost identical across positions \u2014 {hi.position} and "
            f"{lo.position}, the highest and lowest paid roles, differ by only "
            f"{abs(hi.avg_compensation - lo.avg_compensation):,.0f} USD ({abs(pct_diff)}%), "
            f"out of averages around {lo.avg_compensation:,.0f} USD.")
else:
    finding(f"The average {hi.position} earns {pct_diff}% more than the average {lo.position} "
            f"({hi.avg_compensation:,.0f} vs {lo.avg_compensation:,.0f} USD per period as supplied).")

In [ ]:
comp_edu = pd.read_sql("""
    SELECT ed.education_level,
           ROUND(AVG(f.basic_salary + f.allowances), 2) AS avg_compensation
    FROM employee e
    JOIN education ed ON e.employee_id = ed.employee_id
    JOIN finance f  ON e.employee_id = f.employee_id
    GROUP BY ed.education_level ORDER BY avg_compensation DESC
""", con)
order = ['Diploma', "Bachelor's Degree", "Master's Degree", 'Doctoral Degree']
comp_edu = comp_edu.set_index("education_level").reindex(order).reset_index()

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(comp_edu["education_level"], comp_edu["avg_compensation"], color="#7B2CBF")
ax.set_ylabel("Average compensation, USD per period as supplied")
ax.set_title("Average compensation by education level")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

spread = comp_edu["avg_compensation"].max() - comp_edu["avg_compensation"].min()
finding(f"Average compensation varies by only {spread:,.0f} USD across all four education "
        f"levels — education level shows almost no relationship with pay in this data.")

---
## Objective 4 &nbsp;—&nbsp; Health-related operational risk

> Brief: *“Identify health related operational risks.”*

This is administrative insurance and leave data only. Nothing clinical, so nothing diagnostic is
drawn from it — the risk being measured is operational (lapsed cover, low leave balance), not
medical.

In [ ]:
ins_status = pd.read_sql("SELECT insurance_status, COUNT(*) AS n FROM health GROUP BY insurance_status", con)
ins_status["share"] = (ins_status["n"] / ins_status["n"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(4,4))
colors = {"Active":"#06A77D","Pending":"#E0A200","Expired":"#C1121F"}
ax.pie(ins_status["n"], labels=ins_status["insurance_status"], autopct="%1.0f%%",
       colors=[colors[s] for s in ins_status["insurance_status"]])
ax.set_title("Insurance status")
plt.tight_layout(); plt.show()

expired = ins_status.set_index("insurance_status").loc["Expired"]
pending = ins_status.set_index("insurance_status").loc["Pending"]
finding(f"{round(expired.share + pending.share, 1)}% of employees do not currently have Active "
        f"insurance cover — {expired.share}% Expired and {pending.share}% Pending — "
        f"more than a third of the workforce.")

In [ ]:
risk_dept = pd.read_sql("""
    SELECT d.department_name,
        SUM(CASE WHEN h.insurance_status = 'Expired' THEN 1 ELSE 0 END) AS expired,
        SUM(CASE WHEN h.insurance_status = 'Pending' THEN 1 ELSE 0 END) AS pending,
        COUNT(*) AS headcount
    FROM employee e
    JOIN department d ON e.department_code = d.department_code
    JOIN health h ON e.employee_id = h.employee_id
    GROUP BY d.department_name
""", con)
risk_dept["at_risk_pct"] = ((risk_dept["expired"] + risk_dept["pending"]) / risk_dept["headcount"] * 100).round(1)
risk_dept = risk_dept.sort_values("at_risk_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8,4))
ax.barh(risk_dept["department_name"], risk_dept["at_risk_pct"], color="#C1121F")
ax.invert_yaxis()
ax.set_xlabel("% Expired or Pending insurance")
ax.set_title("Insurance risk by department")
plt.tight_layout(); plt.show()

worst = risk_dept.iloc[0]; best = risk_dept.iloc[-1]
finding(f"{worst.department_name} has the highest insurance risk at {worst.at_risk_pct}% "
        f"Expired or Pending, {round(worst.at_risk_pct - best.at_risk_pct, 1)} points higher "
        f"than {best.department_name}, the lowest.")

In [ ]:
LOW = 5   # days or fewer counts as a low balance
low_balance = pd.read_sql(f"SELECT COUNT(*) AS n FROM health WHERE medical_leave_balance <= {LOW}", con).iloc[0,0]
total_h = pd.read_sql("SELECT COUNT(*) AS n FROM health", con).iloc[0,0]

fig, ax = plt.subplots(figsize=(7,4))
bal = pd.read_sql("SELECT medical_leave_balance FROM health", con)
ax.hist(bal["medical_leave_balance"], bins=31, color="#4361EE", edgecolor="white")
ax.axvline(LOW, color="#C1121F", linestyle="--", label=f"{LOW} days or fewer")
ax.set_xlabel("Medical leave balance, days"); ax.set_ylabel("Employees")
ax.set_title("Medical leave balance distribution"); ax.legend()
plt.tight_layout(); plt.show()

finding(f"{low_balance:,} employees ({round(low_balance/total_h*100,1)}%) have {LOW} days or "
        f"fewer of medical leave remaining, out of a possible 30 — worth flagging for HR "
        f"attention regardless of department.")

---
## Objective 5 &nbsp;—&nbsp; Department efficiency

> Brief: *“Identify inefficiencies in departmental staffing and spend”* and *“Assess ... cost
> distribution.”*

This is where `department_performance` is used correctly: at its own grain, department-year,
never joined onto individual employees. Revenue per employee divides a department total by a
department headcount — both taken at the department level, so nothing is repeated or diluted.

In [ ]:
dept_eff = pd.read_sql("""
    SELECT department_name, year, average_performance_score,
           total_revenue_generated, total_cost, training_hours_completed,
           (total_revenue_generated - total_cost) AS net_margin
    FROM department_performance
    ORDER BY department_name, year
""", con)
dept_eff.head(10)

In [ ]:
y2025 = dept_eff[dept_eff.year == 2025].copy()

headcount = pd.read_sql("""
    SELECT d.department_name, COUNT(*) AS headcount
    FROM employee e JOIN department d ON e.department_code = d.department_code
    GROUP BY d.department_name
""", con)
y2025 = y2025.merge(headcount, on="department_name")
y2025["revenue_per_employee"] = (y2025["total_revenue_generated"] / y2025["headcount"]).round(0)
y2025 = y2025.sort_values("net_margin", ascending=False)
y2025[["department_name","total_revenue_generated","total_cost","net_margin","headcount","revenue_per_employee"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
colors = ["#06A77D" if v > 0 else "#C1121F" for v in y2025["net_margin"]]
ax.barh(y2025["department_name"], y2025["net_margin"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("Revenue minus cost, USD, 2025")
ax.set_title("Department net margin, 2025")
plt.tight_layout(); plt.show()

best = y2025.iloc[0]; worst = y2025.iloc[-1]
finding(f"In 2025, {best.department_name} produced the largest net margin at "
        f"{best.net_margin:,.0f} USD, while {worst.department_name} produced the smallest "
        f"at {worst.net_margin:,.0f} USD — a gap of {best.net_margin - worst.net_margin:,.0f}.")

In [ ]:
y2025_rpe = y2025.sort_values("revenue_per_employee", ascending=False)

fig, ax = plt.subplots(figsize=(8,4))
ax.barh(y2025_rpe["department_name"], y2025_rpe["revenue_per_employee"], color="#0FA3A3")
ax.invert_yaxis()
ax.set_xlabel("Revenue per employee, USD, 2025")
ax.set_title("Revenue per employee by department, 2025")
plt.tight_layout(); plt.show()

top = y2025_rpe.iloc[0]; bottom = y2025_rpe.iloc[-1]
ratio = round(top.revenue_per_employee / bottom.revenue_per_employee, 1)
finding(f"{top.department_name} generates {ratio}x more revenue per employee than "
        f"{bottom.department_name} ({top.revenue_per_employee:,.0f} vs "
        f"{bottom.revenue_per_employee:,.0f} USD) — the clearest efficiency gap in the data.")

In [ ]:
trend = dept_eff.pivot_table(index="year", columns="department_name", values="average_performance_score")

fig, ax = plt.subplots(figsize=(9,4.5))
for col in trend.columns:
    ax.plot(trend.index, trend[col], marker="o", label=col, linewidth=1.6)
ax.set_xlabel("Year"); ax.set_ylabel("Average performance score")
ax.set_title("Department performance score, 2021-2025")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1,1))
plt.tight_layout(); plt.show()

swing = (trend.max() - trend.min()).sort_values(ascending=False)
most_volatile = swing.index[0]
finding(f"{most_volatile} shows the largest swing in average performance score across the five "
        f"years, ranging {trend[most_volatile].min()} to {trend[most_volatile].max()} — "
        f"more volatile than any other department.")

---
## Findings list

Every finding generated above, collected in one place. This is what gets copied into the
Stage 9 analysis report.

In [ ]:
for i, f in enumerate(findings, 1):
    print(f"{i:>2}. {f}")
print(f"\n{len(findings)} findings generated")

---
## A note on Objective 2

The education-versus-placement mismatch figure in this notebook depends entirely on the
`FIELD_MATCHES_DEPARTMENT` mapping defined above, which is my judgement, not a fact supplied by
the data or the client. Before this number goes into the final report, it should be reviewed —
ideally by the client, who knows what field of study actually qualifies someone for each
department at Roblox. A different mapping could reasonably move the mismatch rate by 10 points
or more in either direction.

### What comes next

Stage 8 builds the executive dashboard from these same queries. Stage 9 turns the findings list
above into the analysis report and the recommendations.